# SQ-1c — Canary audit + empirical ε lower bound

A **privacy audit** is not the same as an MIA. Where SQ-1's per-record MIA
measures how identifiable naturally-occurring training records are, a
*canary audit* deliberately injects maximally-vulnerable constructed
records during FL training and asks whether the released model's behaviour
on those canaries falsifies the chain's declared (ε, δ)-DP claim.

This notebook follows Jagielski–Ullman–Oprea (NeurIPS 2020) and
Nasr et al. (USENIX Security 2023). Canary construction is random-pixel
inputs with arbitrary labels — the worst-case sample the DP definition
must bound. Detection is LiRA against the canaries themselves. The
empirical ε lower bound (Clopper–Pearson) is reported alongside the
chain card's declared ε design target.

In [1]:
from pathlib import Path
import json, time
import numpy as np

REPO_ROOT = Path.cwd().parent.parent if Path.cwd().name == "sq1-calibration" else Path.cwd()
print("REPO_ROOT =", REPO_ROOT)

import sys; sys.path.insert(0, str(REPO_ROOT))
from flta_eval import datasets, fl, attacks
from flta_eval.audit import empirical_epsilon_lower_bound, write_result_record

PAPER_SCALE = True
MASTER_SEED = 20260525

# Canary construction parameters.
N_CANARIES   = 200   # how many canaries to inject
N_NEG_TRIALS = 1000 # negative-sample trials for FPR estimation
DELTA        = 1e-5

REPO_ROOT = /Users/john.smith/Documents/dev/stepwise-privacy-cards


In [2]:
ds = datasets.load_bloodmnist(REPO_ROOT / "data")
manifest = datasets.load_partition(REPO_ROOT / "data")
pod_data = [(ds["X_train"][np.array(manifest["pod_indices"][i])],
             ds["y_train"][np.array(manifest["pod_indices"][i])])
            for i in range(manifest["n_pods"])]

input_dim = ds["input_dim"]
n_classes = ds["n_classes"]
print(f"FL pods: {len(pod_data)}; input_dim={input_dim}; n_classes={n_classes}")

FL pods: 50; input_dim=2352; n_classes=8


## Construct canaries — uniform-pixel inputs, arbitrary labels

A DP guarantee with parameter ε bounds the worst-case adversary's
distinguishing advantage between *any* two adjacent datasets. The hardest
sample to bound is an out-of-distribution input whose presence the
model can memorise without disturbing the rest of the loss landscape.
Random-pixel inputs are a standard canary choice (Carlini–Tramèr et al.
"extracting training data from large language models" 2021; Maddock–Eilbing
2024).

In [3]:
rng = np.random.default_rng(MASTER_SEED + 999)
canary_X = rng.random((N_CANARIES, input_dim)).astype(np.float32)
canary_y = rng.integers(0, n_classes, size=N_CANARIES, dtype=np.int64)
print(f"Built {N_CANARIES} canaries; label histogram:", np.bincount(canary_y, minlength=n_classes).tolist())

Built 200 canaries; label histogram: [25, 22, 26, 26, 31, 27, 21, 22]


## Train FL with canaries inserted into one pod

We inject the canaries into pod 0 — "one client volunteers a small
batch of maximally-vulnerable records". The training loop is the same
DP-SGD FedAvg used by SQ-1; canary presence is the only difference.

In [4]:
Xp0, yp0 = pod_data[0]
pod_data_canary = list(pod_data)
pod_data_canary[0] = (np.vstack([Xp0, canary_X]),
                      np.concatenate([yp0, canary_y]))
print(f"Pod 0 size: {len(yp0)} -> {len(pod_data_canary[0][1])} after canary injection")

cfg = fl.FLConfig(n_rounds=20, client_lr=0.1, client_batch_size=32,
                  clip_norm=1.0, noise_multiplier=1.1)
model = fl.MLP(input_dim=input_dim, hidden_dim=64, n_classes=n_classes)
model.init_from_seed(MASTER_SEED, "fl.canary_audit.train")

t0 = time.time()
trained, hist = fl.federated_train(model=model, pod_data=pod_data_canary,
                                   X_test=ds["X_test"], y_test=ds["y_test"],
                                   config=cfg, master_seed=MASTER_SEED,
                                   namespace="fl.canary_audit.train")
fl_s = time.time() - t0
print(f"FL trained with canaries in {fl_s:.1f}s; test accuracy {trained.accuracy(ds["X_test"], ds["y_test"]):.3f}")

Pod 0 size: 251 -> 451 after canary injection


FL trained with canaries in 0.8s; test accuracy 0.253


## Measure canary detection under LiRA at FPR = 10⁻³

For each canary, fit per-target Gaussians to shadow IN / OUT logits and
test whether the released model crosses the per-canary threshold. The
positive-trial detection rate is the empirical canary TPR; the matching
negative trials come from non-canary references drawn from the same pool.

In [5]:
# Build a "pod_data" that contains the canaries as a synthetic pod for the MIA
# sampler. The MIA picks targets from the pool; we want all targets to be canaries.
canary_pod = (canary_X, canary_y)
pod_data_for_mia = pod_data_canary + [canary_pod]
canary_pod_id = len(pod_data_canary)  # last pod

n_shadow = 128 if PAPER_SCALE else 32

t0 = time.time()
mia_canary = attacks.per_record_mia(
    federated_model=trained, pod_data=pod_data_for_mia,
    X_test=ds["X_test"], y_test=ds["y_test"],
    n_targets=N_CANARIES,
    n_shadow_runs=n_shadow, shadow_steps=15,
    fpr_targets=(0.001,), meta_classifier="lira",
    n_bootstrap=500,
    master_seed=MASTER_SEED, namespace="attacks.canary_audit.lira",
)
mia_s = time.time() - t0
print(f"Canary MIA: {mia_s:.1f}s, {mia_canary.n_targets_swept} canaries swept")
print(f"  worst-canary TPR @ FPR=1e-3:    {mia_canary.worst_record_tpr_at_fpr:.4f}")
print(f"  worst-canary TPR 95%% CI:        [{mia_canary.worst_record_tpr_ci_lower:.4f}, {mia_canary.worst_record_tpr_ci_upper:.4f}]")
print(f"  mean canary TPR @ FPR=1e-3:     {np.mean([t["tpr_at_fpr"] for t in mia_canary.per_target]):.4f}")

Canary MIA: 134.6s, 200 canaries swept
  worst-canary TPR @ FPR=1e-3:    0.1456
  worst-canary TPR 95%% CI:        [0.0306, 0.3556]
  mean canary TPR @ FPR=1e-3:     0.0082


## Empirical ε lower bound

Count canary detections (positive trials) and non-canary detections
(negative trials). The Clopper-Pearson lower bound on canary TPR + upper
bound on FPR + Jagielski-Ullman-Oprea "Auditing differentially private
machine learning" (NeurIPS 2020) Eq. 4 give a one-sided lower bound on ε.

In [6]:
canary_detections = sum(t["released_model_hit"] for t in mia_canary.per_target)

# Matched negative trials: train FL_OUT *without* the canaries; check whether
# each canary crosses its per-target LiRA threshold under the OUT model.
# This follows Nasr et al. (USENIX Security 2023) and gives an FPR matched
# to the canary distribution itself, not to natural pool samples.

model_out = fl.MLP(input_dim=input_dim, hidden_dim=64, n_classes=n_classes)
model_out.init_from_seed(MASTER_SEED, "fl.canary_audit.out.train")
t0 = time.time()
trained_out, _ = fl.federated_train(model=model_out, pod_data=pod_data,  # NO canaries
                                    X_test=ds["X_test"], y_test=ds["y_test"],
                                    config=cfg, master_seed=MASTER_SEED,
                                    namespace="fl.canary_audit.out.train")
print(f"FL_OUT trained in {time.time()-t0:.1f}s; test accuracy {trained_out.accuracy(ds["X_test"], ds["y_test"]):.3f}")

# Evaluate each canary against its OWN per-target threshold under the OUT model
canary_thresholds = np.array([t["threshold_at_fpr"] for t in mia_canary.per_target])
out_signals = trained_out.predict_proba(canary_X)[np.arange(N_CANARIES), canary_y]
eps_clip = 1e-7
out_logits = np.log(out_signals.clip(eps_clip, 1 - eps_clip) /
                    (1 - out_signals).clip(eps_clip, 1 - eps_clip))

false_positives = int((out_logits >= canary_thresholds).sum())
N_NEG_TRIALS = N_CANARIES  # matched negative trial count

audit = empirical_epsilon_lower_bound(
    n_positives=canary_detections, n_positive_trials=N_CANARIES,
    n_false_positives=false_positives, n_negative_trials=N_NEG_TRIALS,
    delta=DELTA,
)
print(f"Canary detections (TP) under FL_IN:   {canary_detections}/{N_CANARIES}")
print(f"Canary detections (FP) under FL_OUT:  {false_positives}/{N_NEG_TRIALS}")
print(f"TPR lower (Clopper-Pearson 95%%):     {audit["tpr_lower"]:.4f}")
print(f"FPR upper (Clopper-Pearson 95%%):     {audit["fpr_upper"]:.4f}")
print(f"Empirical ε lower bound:                  {audit["epsilon_lower_bound"]:.3f}")
print(f"Feasible (TPR_lower > δ):              {audit["feasible"]}")

FL_OUT trained in 0.8s; test accuracy 0.149
Canary detections (TP) under FL_IN:   36/200
Canary detections (FP) under FL_OUT:  34/200
TPR lower (Clopper-Pearson 95%%):     0.1366
FPR upper (Clopper-Pearson 95%%):     0.2198
Empirical ε lower bound:                  0.000
Feasible (TPR_lower > δ):              True


## Compare with the card's declared ε design target

The card declares ε = 2.0 as design target. The PRV-accountant ε at the
configured (σ=1.1, T=20, q=1.0) is 24.92; the RDP envelope is 44.57.
The empirical ε lower bound from the canary audit is a third
quantity — what the *released model actually leaks under LiRA*. If
empirical ε_lower exceeds the card's declared ε, the card's privacy
claim is falsified by direct measurement.

In [7]:
card = json.loads((REPO_ROOT / "card" / "example_chain.json").read_text())
declared_eps = card["pet_components"][2]["parameters"]["epsilon_design_target"]
measured_rdp = card["risk_calibration"]["measured_epsilon"]["rdp_envelope"]
measured_prv = card["risk_calibration"]["measured_epsilon"]["prv_gopi_2021"]

print("=== ε comparison ===")
print(f"  Card design target:        {declared_eps:.2f}")
print(f"  Measured ε (RDP envelope): {measured_rdp:.2f}")
print(f"  Measured ε (PRV):          {measured_prv:.2f}")
print(f"  Empirical ε lower bound:   {audit['epsilon_lower_bound']:.2f}  (canary audit at δ={DELTA})")
print()
if audit["feasible"] and audit["epsilon_lower_bound"] > declared_eps:
    print(f"** Empirical ε ({audit['epsilon_lower_bound']:.2f}) exceeds the card's design target ({declared_eps:.2f}).")
    print(f"** The audit *falsifies* the design-target ε.")
elif audit["feasible"]:
    print(f"   Empirical ε ({audit['epsilon_lower_bound']:.2f}) is at or below the declared design target ({declared_eps:.2f}).")
    print(f"   The audit does not falsify the design-target ε.")
else:
    print("   Canary audit did not produce a feasible ε lower bound (TPR_lower ≤ δ).")
    print("   This typically means more canaries / more shadow runs are needed.")

# Write the audit record.
record = write_result_record(
    target_dir=REPO_ROOT / "results" / "sq1",
    attack="canary_audit", variant="SQ-1c", threat_profile="R",
    dataset={"name": "bloodmnist", "n_canaries": N_CANARIES, "n_negative_trials": N_NEG_TRIALS,
             "sha256": json.loads((REPO_ROOT / "data" / "_manifest.json").read_text())["npz_sha256"]},
    config={"paper_scale": PAPER_SCALE, "n_shadow_runs": n_shadow, "sigma": cfg.noise_multiplier,
            "clip_norm": cfg.clip_norm, "n_rounds": cfg.n_rounds, "sample_rate": 1.0, "delta": DELTA},
    seed_namespace="attacks.canary_audit.bloodmnist.v1",
    result={"canary_detections": int(canary_detections),
            "empirical_fpr": false_positives / N_NEG_TRIALS,
            "worst_canary_tpr": float(mia_canary.worst_record_tpr_at_fpr),
            "worst_canary_tpr_ci_lower": float(mia_canary.worst_record_tpr_ci_lower),
            "worst_canary_tpr_ci_upper": float(mia_canary.worst_record_tpr_ci_upper),
            **audit, "declared_epsilon_design_target": declared_eps,
            "measured_epsilon_rdp": measured_rdp, "measured_epsilon_prv": measured_prv,
            "audit_falsifies_design_target": bool(audit["feasible"] and audit["epsilon_lower_bound"] > declared_eps)},
    tolerances={"note": "Empirical ε lower bound; falsification iff > declared design target."},
)
print(f"Audit record written: {record.relative_to(REPO_ROOT)}")

=== ε comparison ===
  Card design target:        2.00
  Measured ε (RDP envelope): 44.57
  Measured ε (PRV):          24.92
  Empirical ε lower bound:   0.00  (canary audit at δ=1e-05)

   Empirical ε (0.00) is at or below the declared design target (2.00).
   The audit does not falsify the design-target ε.
Audit record written: results/sq1/canary_audit__SQ-1c__2026-05-26T07-29-03Z.json
